# 🚗 Custom LPRNet Training for Philippine License Plates

This notebook is designed to run on **Google Colab** to train your own LPRNet model using the `LPRnet-keras` repository architecture.

### Instructions:
1. **Upload your `LPRnet-keras` folder** to your Google Drive.
2. **Upload your generated dataset** to Google Drive. The dataset should consist of images that have already been cropped and preprocessed (CLAHE + Sharpening) as they appear in the production pipeline.
3. Set your runtime type to **GPU** (`Runtime > Change runtime type > T4 GPU`).
4. Update the paths in the cells below to match your Google Drive structure.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import sys
import cv2
import glob
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

# Ensure TensorFlow uses the GPU
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

# IMPORTANT: Add the LPRnet-keras folder to the Python path
# Change this path to where you uploaded your C:\LPRnet-keras folder in Drive
LPRNET_REPO_PATH = '/content/drive/MyDrive/LPRnet-keras'
sys.path.append(LPRNET_REPO_PATH)

try:
    from LPRnet.LPRnet_separable import LPRnet, CTCLoss, global_context
    print("Successfully imported LPRNet architecture!")
except ImportError as e:
    print(f"Import Error: {e}")
    print("Make sure LPRNET_REPO_PATH correctly points to the LPRnet-keras folder.")

## 1. Configuration & Vocabulary
Define the target image shape and the characters available on Philippine license plates.

In [ ]:
IMAGE_SHAPE = (94, 24)  # (Width, Height)

# Vocabulary definition. 
# Standard PH plates use all letters and numbers. 
# Note: The original LPRnet-keras excluded 'I' and 'O' to avoid confusion with '1' and '0'.
# If your dataset contains I and O, include them here.
CHARS = "ABCDEFGHIJKLMNPQRSTUVWXYZ0123456789"
CHARS_DICT = {char: i for i, char in enumerate(CHARS)}
NUM_CLASS = len(CHARS) + 1  # +1 for the CTC Blank token

BATCH_SIZE = 64
EPOCHS = 100

## 2. Load and Preprocess Dataset
Since your dataset images are already cropped and visually preprocessed (CLAHE + Sharpened) based on your extraction pipeline, we only need to ensure they are the correct size `(94x24)`, converted to `RGB`, and normalized to `[0, 1]`.

In [ ]:
def load_dataset(data_dir):
    images = []
    labels = []
    
    # Supports .png and .jpg images
    file_paths = glob.glob(os.path.join(data_dir, "*.png")) + glob.glob(os.path.join(data_dir, "*.jpg"))
    print(f"Found {len(file_paths)} images in {data_dir}")
    
    for file_path in file_paths:
        # Extract label from filename. Example: "ABC1234_01.png" -> "ABC1234"
        basename = os.path.basename(file_path)
        label_str = basename.split('_')[0].split('-')[0].upper()
        
        # If you excluded 'I' and 'O' in CHARS, map them to '1' and '0' respectively:
        # label_str = label_str.replace("O", "0").replace("I", "1")
        
        try:
            label = [CHARS_DICT[char] for char in label_str]
        except KeyError as e:
            print(f"Skipping {basename}: character {e} not in CHARS definition.")
            continue
            
        img = cv2.imread(file_path)
        if img is None:
            continue
            
        # Resize to strict dimensions (Width=94, Height=24)
        img = cv2.resize(img, IMAGE_SHAPE, interpolation=cv2.INTER_CUBIC)
        
        # OpenCV loads as BGR, LPRNet expects RGB
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Normalize pixel values to [0, 1] as standard Keras floats
        img = img.astype(np.float32) / 255.0
        
        images.append(img)
        labels.append(label)
        
    return np.array(images, dtype=np.float32), labels

# --- UPDATE THESE PATHS TO YOUR DATASET DIRECTORIES IN DRIVE ---
TRAIN_DIR = '/content/drive/MyDrive/dataset/train'
VAL_DIR = '/content/drive/MyDrive/dataset/val'

# Uncomment and run this once you have updated the paths above:
# X_train, y_train = load_dataset(TRAIN_DIR)
# X_val, y_val = load_dataset(VAL_DIR)
# print(f"Loaded {len(X_train)} training samples and {len(X_val)} validation samples.")

## 3. Prepare TensorFlow Dataset Pipeline

In [ ]:
def create_tf_dataset(X, y, batch_size, is_training=True):
    # CTC Loss requires handling variable length sequences (e.g. 6 or 7 characters)
    ragged_y = tf.ragged.constant(y).to_tensor()
    
    dataset = tf.data.Dataset.from_tensor_slices((X, ragged_y))
    if is_training:
        dataset = dataset.shuffle(buffer_size=min(len(X), 10000)).repeat()
    else:
        dataset = dataset.repeat()
        
    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

# Uncomment after loading data:
# train_dataset = create_tf_dataset(X_train, y_train, BATCH_SIZE, is_training=True)
# val_dataset = create_tf_dataset(X_val, y_val, BATCH_SIZE, is_training=False)
# 
# steps_per_epoch = max(1, len(X_train) // BATCH_SIZE)
# validation_steps = max(1, len(X_val) // BATCH_SIZE)

## 4. Build and Compile Model

In [ ]:
# Initialize the architecture
model = LPRnet()

# Compile with CTC Loss
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3), 
    loss=CTCLoss
)

# Build model with shape (Batch, Height, Width, Channels)
model.build((None, 24, 94, 3))
model.summary()

## 5. Training Loop

In [ ]:
callbacks = [
    # Save the best weights based on validation loss
    keras.callbacks.ModelCheckpoint(
        "/content/drive/MyDrive/ph_lprnet_best.h5", 
        monitor="val_loss", 
        save_best_only=True, 
        save_weights_only=True,
        verbose=1
    ),
    # Reduce Learning Rate if validation loss plateaus
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', 
        factor=0.5, 
        patience=5, 
        min_lr=1e-5,
        verbose=1
    ),
    # Stop training early if no improvement
    keras.callbacks.EarlyStopping(
        monitor='val_loss', 
        patience=15, 
        restore_best_weights=True,
        verbose=1
    )
]

# Uncomment to start training:
# history = model.fit(
#     train_dataset,
#     epochs=EPOCHS,
#     steps_per_epoch=steps_per_epoch,
#     validation_data=val_dataset,
#     validation_steps=validation_steps,
#     callbacks=callbacks
# )

## 6. Evaluate and Plot History

In [ ]:
# Uncomment to plot:
# plt.figure(figsize=(10, 5))
# plt.plot(history.history['loss'], label='Train Loss')
# plt.plot(history.history['val_loss'], label='Val Loss')
# plt.title('LPRNet Training History')
# plt.xlabel('Epoch')
# plt.ylabel('CTC Loss')
# plt.legend()
# plt.grid(True)
# plt.show()

## 7. Export to TFLite
Once training is complete, convert the Keras model to TensorFlow Lite format for deployment on Edge devices (like Raspberry Pi) or just for faster inference in your production pipeline.

In [ ]:
def export_to_tflite(keras_model, output_path="/content/drive/MyDrive/ph001_custom.tflite"):
    print("Converting model to TFLite...")
    converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)
    
    # Optional: Enable optimization to reduce model size and improve speed
    # converter.optimizations = [tf.lite.Optimize.DEFAULT]
    
    tflite_model = converter.convert()
    
    with open(output_path, "wb") as f:
        f.write(tflite_model)
        
    print(f"Model successfully converted and saved to {output_path}")

# Uncomment to export:
# export_to_tflite(model)